# 03 — Operations & Communication
**AGENTIC X | Studio 67 — CAI461**



## Cell 1 — Setup current GitHub repository

In [1]:
from pathlib import Path
import os, sys, subprocess
REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").exists():
    raise RuntimeError("/content/AgenticX exists but is not a Git checkout")
else:
    changes = subprocess.run(["git", "-C", str(ROOT), "status", "--porcelain"],
                             capture_output=True, text=True, check=True).stdout.strip()
    if changes:
        print("Local changes detected; no git pull was performed to avoid overwriting them.")
        print("For the final evaluation, use a fresh Colab runtime or update the checkout safely.")
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Repository ready:", ROOT)

Repository ready: /content/AgenticX


## Cell 2 — Imports & verify Operations V4

In [2]:
import importlib
import pandas as pd
from IPython.display import display
from core.environment import load_dataset, get_case, clean, HIDDEN_CASE_FIELDS, HIDDEN_OPTION_FIELDS
from core.models import AgentEvaluation, ACLMessage
import agents.operations as operations_module
import core.communication as communication_module
importlib.reload(operations_module)
importlib.reload(communication_module)
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent

OperationsAgent = operations_module.OperationsAgent
MessageBus = communication_module.MessageBus
EXPECTED = "2026-11-operations-gemini-all-v4"
assert OperationsAgent.VERSION == EXPECTED, (
    f"Expected {EXPECTED}, got {OperationsAgent.VERSION!r}. "
    "Use latest GitHub files / fresh Colab runtime before running this notebook."
)
assert communication_module.ONTOLOGY == "studio67.event_crisis"
assert Path(operations_module.__file__).resolve() == ROOT / "agents/operations.py"
assert Path(communication_module.__file__).resolve() == ROOT / "core/communication.py"
assert set(communication_module.VALID_PERFORMATIVES) == {
    'REQUEST','INFORM','CFP','PROPOSE','ACCEPT','REFUSE'
}
print("Verified Operations V4 + current ACL module:", operations_module.__file__, communication_module.__file__, sep="\n")

Verified Operations V4 + current ACL module:
/content/AgenticX/agents/operations.py
/content/AgenticX/core/communication.py


## Cell 3 — Dataset and real scenario inputs

In [3]:
cases_df, options_df = load_dataset()
assert len(cases_df) == 150 and len(options_df) == 450
assert set(cases_df['scenario']) == {'Screen Failure', 'Budget Overrun', 'Safety Conflict'}
case, options = get_case('SCR-001')
assert len(options) == 3
assert all(not (HIDDEN_OPTION_FIELDS & set(option)) for option in options)
assert not (HIDDEN_CASE_FIELDS & set(case))
print('Cases:',len(cases_df),'| Options:',len(options_df))
print('Example:',case['case_id'],case['incident'])
display(pd.DataFrame(options)[['option_id','option_name','duration_min','crew_required','needs_backup','supplier_available']])

Cases: 150 | Options: 450
Example: SCR-001 Main LED screen failed during final rehearsal


,option_id,option_name,duration_min,crew_required,needs_backup,supplier_available
0,SCR-001-O1,Use backup LED,40,1,True,True
1,SCR-001-O2,Rent replacement screen,70,1,False,True
2,SCR-001-O3,Continue with projection,25,1,False,True


## Cell 4 — Operations PEAS and Hybrid classification

In [4]:
operations_peas = pd.DataFrame([
    {'Component':'Performance Measure','Description':'Practical solutions satisfying time, crew, equipment, venue and supplier limits'},
    {'Component':'Environment','Description':'Studio 67 simulated event crises, dynamic remaining time'},
    {'Component':'Actuators','Description':'Approve/refuse an option, score, justify decision, send ACL messages'},
    {'Component':'Sensors','Description':'Incident, option duration, staff counts, backup, supplier, venue and rule-check results'},
])
display(operations_peas)
print('Hybrid Agent: mandatory deterministic resource checks + Gemini operational judgment.')
print('Gemini is called for BOTH passing and failing resource checks; failed gates always veto approval.')

,Component,Description
0,Performance Measure,"Practical solutions satisfying time, crew, equ..."
1,Environment,"Studio 67 simulated event crises, dynamic rema..."
2,Actuators,"Approve/refuse an option, score, justify decis..."
3,Sensors,"Incident, option duration, staff counts, backu..."


Hybrid Agent: mandatory deterministic resource checks + Gemini operational judgment.
Gemini is called for BOTH passing and failing resource checks; failed gates always veto approval.


## Cell 5 — Operations variables and parameters

In [5]:
variables = pd.DataFrame([
    ('time_left_min','Case','Minutes remaining','Mandatory deadline check; failed check means rejection'),
    ('duration_min','Option','Implementation duration','Compared with remaining time'),
    ('crew_count_available','Case','Available workers','Must cover crew_required when needed'),
    ('crew_available','Case','Staff availability flag','Must be True for crew-dependent options'),
    ('crew_required / needs_crew','Option','Staff requirement','Determines whether crew check applies'),
    ('backup_available / needs_backup','Case / Option','Equipment availability and demand','Unavailable required backup forces rejection'),
    ('venue_access / requires_venue_access','Case / Option','Venue availability and demand','Missing required access forces rejection'),
    ('supplier_available','Option','Supplier status','Must be available for proposal feasibility'),
    ('AGENTICX_MODEL','Environment','Gemini model identifier','Controls which LLM assesses practical feasibility'),
], columns=['Variable','Source','Purpose','Effect'])
display(variables)

,Variable,Source,Purpose,Effect
0,time_left_min,Case,Minutes remaining,Mandatory deadline check; failed check means r...
1,duration_min,Option,Implementation duration,Compared with remaining time
2,crew_count_available,Case,Available workers,Must cover crew_required when needed
3,crew_available,Case,Staff availability flag,Must be True for crew-dependent options
4,crew_required / needs_crew,Option,Staff requirement,Determines whether crew check applies
5,backup_available / needs_backup,Case / Option,Equipment availability and demand,Unavailable required backup forces rejection
6,venue_access / requires_venue_access,Case / Option,Venue availability and demand,Missing required access forces rejection
7,supplier_available,Option,Supplier status,Must be available for proposal feasibility
8,AGENTICX_MODEL,Environment,Gemini model identifier,Controls which LLM assesses practical feasibility


## Cell 6 — Verify operational constraints without Gemini

In [6]:
base_case, base_options = get_case('SCR-001')
base_option = base_options[0]
assert OperationsAgent.check_constraints(base_case, base_option)[0], 'Expected the first SCR-001 option to pass basic resource gates'

constraint_tests = [
    ('Missing time', {**base_case,'time_left_min':None}, base_option,'time'),
    ('Negative time', {**base_case,'time_left_min':-5}, base_option,'time'),
    ('Insufficient time', {**base_case,'time_left_min':0}, base_option,'time'),
    ('Missing duration',base_case,{**base_option,'duration_min':None},'time'),
    ('Insufficient crew',{**base_case,'crew_count_available':0},base_option,'crew'),
    ('Crew unavailable',{**base_case,'crew_available':False},base_option,'crew'),
    ('Backup unavailable',{**base_case,'backup_available':False},base_option,'backup'),
    ('Venue closed',{**base_case,'venue_access':False},base_option,'venue_access'),
    ('Supplier unavailable',base_case,{**base_option,'supplier_available':False},'supplier'),
    ('Supplier unknown',base_case,{**base_option,'supplier_available':None},'supplier'),
]
rows=[]
for label,c,o,key in constraint_tests:
    passed,failed,values=OperationsAgent.check_constraints(c,o)
    assert not passed and key in failed, (label,failed)
    rows.append({'Test':label,'Mandatory checks passed':passed,'Failed checks':', '.join(failed)})
display(pd.DataFrame(rows))
print('Validated',len(constraint_tests),'constraint failures with no API calls')

,Test,Mandatory checks passed,Failed checks
0,Missing time,False,time
1,Negative time,False,time
2,Insufficient time,False,time
3,Missing duration,False,time
4,Insufficient crew,False,crew
5,Crew unavailable,False,crew
6,Backup unavailable,False,backup
7,Venue closed,False,venue_access
8,Supplier unavailable,False,supplier
9,Supplier unknown,False,supplier


Validated 10 constraint failures with no API calls


## Cell 7 — V4 contract: LLM evaluates both outcomes, never overrides mandatory veto

In [7]:
class FakeGemini:
    """Small deterministic substitute; verifies the real evaluate() call path at zero API cost."""
    def __init__(self,approved=True,score=88):
        self.prompts=[]
        self.approved=approved
        self.score=score
    def invoke(self,prompt):
        self.prompts.append(prompt)
        return {'approved':self.approved,'score':self.score,
                'reason':'Operational assessment based on supplied resources and timing.'}

fake=FakeGemini(approved=True,score=88)
agent=OperationsAgent(llm=fake)
positive=agent.evaluate(base_case,base_option)
assert isinstance(positive,AgentEvaluation)
assert positive.agent=='Operations' and positive.option_id==base_option['option_id']
assert positive.approved and positive.score==88
assert len(fake.prompts)==1 and 'MANDATORY_CHECKS_PASSED: True' in fake.prompts[-1]

rows=[]
for label,c,o,key in constraint_tests:
    previous=len(fake.prompts)
    result=agent.evaluate(c,o)
    assert len(fake.prompts)==previous+1, 'Operations V4 must call Gemini for rejected options too'
    assert not result.approved and result.blocking
    assert result.score<=49, 'Infeasible options must not receive an approval-range score'
    assert 'Verified blocking constraints:' in result.reason
    assert key in OperationsAgent.check_constraints(c,o)[1]
    assert 'MANDATORY_CHECKS_PASSED: False' in fake.prompts[-1]
    rows.append({'Test':label,'LLM called':True,'Approved':result.approved,'Score':result.score})
display(pd.DataFrame(rows))
print('V4 passed:',len(fake.prompts),'fake Gemini calls; rule-based veto always enforced.')

,Test,LLM called,Approved,Score
0,Missing time,True,False,49.0
1,Negative time,True,False,49.0
2,Insufficient time,True,False,49.0
3,Missing duration,True,False,49.0
4,Insufficient crew,True,False,49.0
5,Crew unavailable,True,False,49.0
6,Backup unavailable,True,False,49.0
7,Venue closed,True,False,49.0
8,Supplier unavailable,True,False,49.0
9,Supplier unknown,True,False,49.0


V4 passed: 11 fake Gemini calls; rule-based veto always enforced.


## Cell 8 — Independent Gemini rejection with all mandatory checks passed (mocked)

In [8]:
reject_llm=FakeGemini(approved=False,score=25)
reject_agent=OperationsAgent(llm=reject_llm)
refusal=reject_agent.evaluate(base_case,base_option)
assert not refusal.approved and refusal.score==25
assert 'All mandatory operational checks passed.' in refusal.reason
assert 'Operations rejects this proposal.' in refusal.reason
assert len(reject_llm.prompts)==1
print('Hybrid reasoning verified: LLM may reject even when deterministic checks pass.')

Hybrid reasoning verified: LLM may reject even when deterministic checks pass.


## Cell 9 — Test all three scenario categories using the V4 decision path (mocked)

In [9]:
scenarios=[]
for cid in ('SCR-001','BDG-001','SFT-001'):
    c,opts=get_case(cid)
    option=next(o for o in opts if float(o['duration_min'])>0)
    zero_time={**c,'time_left_min':0}
    mock=FakeGemini(approved=True)
    outcome=OperationsAgent(llm=mock).evaluate(zero_time,option)
    assert len(mock.prompts)==1 and not outcome.approved and outcome.blocking
    assert 'Verified blocking constraints:' in outcome.reason
    scenarios.append({'Case':cid,'Scenario':c['scenario'],'Option':option['option_id'],
                      'Gemini mocked calls':len(mock.prompts),'Result':'Rejected by mandatory time constraint'})
display(pd.DataFrame(scenarios))
print('All three scenario categories validated without paid Gemini calls.')

,Case,Scenario,Option,Gemini mocked calls,Result
0,SCR-001,Screen Failure,SCR-001-O1,1,Rejected by mandatory time constraint
1,BDG-001,Budget Overrun,BDG-001-O1,1,Rejected by mandatory time constraint
2,SFT-001,Safety Conflict,SFT-001-O1,1,Rejected by mandatory time constraint


All three scenario categories validated without paid Gemini calls.


## Cell 10 — Audit all 450 options: resource checks ONLY (no Gemini)

In [10]:
rows=[]
for raw_case in cases_df.to_dict('records'):
    c={k:clean(v) for k,v in raw_case.items() if k not in HIDDEN_CASE_FIELDS}
    selected=options_df.loc[options_df['case_id']==c['case_id']]
    for raw_option in selected.to_dict('records'):
        o={k:clean(v) for k,v in raw_option.items() if k not in HIDDEN_OPTION_FIELDS}
        passed,failed,_=OperationsAgent.check_constraints(c,o)
        rows.append({'case_id':c['case_id'],'scenario':c['scenario'],'option_id':o['option_id'],
                     'resource_gates_passed':passed,'failed_checks':', '.join(failed)})
resource_df=pd.DataFrame(rows)
assert len(resource_df)==450 and resource_df['case_id'].nunique()==150
assert resource_df['option_id'].is_unique
display(resource_df.groupby('scenario')['resource_gates_passed'].agg(['count','sum']).reset_index())
print('450 deterministic constraint checks (not 450 LLM evaluations).')

,scenario,count,sum
0,Budget Overrun,150,126
1,Safety Conflict,150,118
2,Screen Failure,150,95


450 deterministic constraint checks (not 450 LLM evaluations).


## Cell 11 — Configure Gemini API securely (live calls start in next cell)

Set `GOOGLE_API_KEY` in Colab Secrets. This section does not print or save your key.

In [11]:
from getpass import getpass
try:
    from google.colab import userdata
    secret=userdata.get('GOOGLE_API_KEY')
except (ImportError,KeyError,RuntimeError):
    secret=None
secret=secret or os.environ.get('GOOGLE_API_KEY')
if not secret:
    secret=getpass('GOOGLE_API_KEY (hidden): ')
if not secret or not secret.strip():
    raise RuntimeError('A Gemini key is required for the live evaluation below')
os.environ['GOOGLE_API_KEY']=secret.strip()
del secret
os.environ.setdefault('AGENTICX_MODEL','gemini-3.5-flash-lite')
operations_agent=OperationsAgent()
print('Live Operations V4 ready; model:',os.environ['AGENTICX_MODEL'])

Live Operations V4 ready; model: gemini-3.5-flash-lite


## Cell 12 — Live Gemini evaluation (feasible and infeasible proposals)

**Two real calls.** The second test confirms that mandatory failed constraints still veto approval after a real Gemini explanation.

In [12]:
case,options=get_case('SCR-001')
live_option=next(o for o in options if OperationsAgent.check_constraints(case,o)[0])
live_approved_or_rejected=operations_agent.evaluate(case,live_option)
assert isinstance(live_approved_or_rejected,AgentEvaluation)
assert live_approved_or_rejected.agent=='Operations'
assert live_approved_or_rejected.option_id==live_option['option_id']
assert 0<=live_approved_or_rejected.score<=100 and live_approved_or_rejected.reason.strip()

infeasible_case={**case,'time_left_min':0}
live_rejected=operations_agent.evaluate(infeasible_case,live_option)
assert isinstance(live_rejected,AgentEvaluation)
assert not live_rejected.approved and live_rejected.blocking
assert live_rejected.score<=49
assert 'Verified blocking constraints:' in live_rejected.reason

display(pd.DataFrame([{'Test':'Feasible checks, Gemini decides',**live_approved_or_rejected.model_dump()},
                      {'Test':'Failed checks, mandatory veto',**live_rejected.model_dump()}]))
print('Real Gemini evaluated both passing and failing options. Mandatory veto verified.')

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,Test,agent,option_id,approved,score,reason,blocking
0,"Feasible checks, Gemini decides",Operations,SCR-001-O1,True,95.0,All mandatory operational checks have successf...,True
1,"Failed checks, mandatory veto",Operations,SCR-001-O1,False,0.0,The proposal must be rejected because the mand...,True


Real Gemini evaluated both passing and failing options. Mandatory veto verified.


## Cell 13 — ACL ontology, messages and validation

In [13]:
bus=MessageBus(case['case_id'])
cfp=bus.send(1,'Coordinator','ALL','CFP','Request operational assessments')
request=bus.send(1,'Coordinator','Operations','REQUEST',f"Evaluate {live_option['option_id']}")
reply=bus.send(1,'Operations','Coordinator','INFORM',live_approved_or_rejected.reason,
               in_reply_to=request.message_id)
assert reply.in_reply_to==request.message_id
assert bus.history(agent='Operations')==[cfp,request,reply]
assert all(m.case_id==case['case_id'] and m.ontology=='studio67.event_crisis' for m in bus.messages)

def expect_error(action):
    try:action()
    except (ValueError,TypeError):return
    raise AssertionError('Expected ACL invalid-input test to fail')

expect_error(lambda:bus.send(0,'Coordinator','Operations','REQUEST','invalid round'))
expect_error(lambda:bus.send(1,'','Operations','REQUEST','missing sender'))
expect_error(lambda:bus.send(1,'Coordinator','Operations','UNKNOWN','unknown performative'))
expect_error(lambda:bus.send(1,'Operations','Coordinator','INFORM','bad reference',in_reply_to='missing'))
expect_error(lambda:bus.send(1,'Safety','Coordinator','INFORM','incorrect author',in_reply_to=request.message_id))
expect_error(lambda:bus.send(1,'Operations','ALL','INFORM','incorrect destination',in_reply_to=request.message_id))
print('ACL ontology, performatives, reply links and invalid-input tests passed.')

ACL ontology, performatives, reply links and invalid-input tests passed.


## Cell 14 — ACL communication with real agent decisions

In [14]:
finance_result=FinanceAgent().evaluate(case,live_option)
safety_result=SafetyAgent().evaluate(case,live_option)
agent_results=[live_approved_or_rejected,finance_result,safety_result]
assert {e.agent for e in agent_results}=={'Operations','Finance','Safety'}

log=MessageBus(case['case_id'])
log.send(1,'Coordinator','ALL','CFP','Request evaluations for '+live_option['option_id'])
log.send(1,'Coordinator','ALL','PROPOSE','Candidate: '+live_option['option_id'])
for evaluation in agent_results:
    req=log.send(1,'Coordinator',evaluation.agent,'REQUEST','Assess '+evaluation.option_id)
    log.send(1,evaluation.agent,'Coordinator','INFORM',evaluation.reason,in_reply_to=req.message_id)
    log.send(1,evaluation.agent,'ALL','ACCEPT' if evaluation.approved else 'REFUSE',
             evaluation.option_id+': '+evaluation.reason)
log.send(1,'Operations','ALL','REFUSE','Controlled impossible-deadline test: '+live_rejected.reason)
if 'ACCEPT' not in {m.performative for m in log.messages}:
    accepted=next((FinanceAgent().evaluate(case,o) for o in options
                   if FinanceAgent().evaluate(case,o).approved),None)
    assert accepted is not None,'Expected at least one budget-feasible option'
    log.send(1,'Finance','ALL','ACCEPT',accepted.reason)
required={'REQUEST','INFORM','CFP','PROPOSE','ACCEPT','REFUSE'}
assert required.issubset({m.performative for m in log.messages})
assert all(m.ontology=='studio67.event_crisis' for m in log.messages)
messages_df=pd.DataFrame(log.to_records())
display(messages_df[['message_id','round','sender','receiver','performative','in_reply_to','ontology','content']])
print('All six ACL performatives represented by real agent/rule decisions.')

,message_id,round,sender,receiver,performative,in_reply_to,ontology,content
0,SCR-001-M0001,1,Coordinator,ALL,CFP,None,studio67.event_crisis,Request evaluations for SCR-001-O1
1,SCR-001-M0002,1,Coordinator,ALL,PROPOSE,None,studio67.event_crisis,Candidate: SCR-001-O1
2,SCR-001-M0003,1,Coordinator,Operations,REQUEST,None,studio67.event_crisis,Assess SCR-001-O1
3,SCR-001-M0004,1,Operations,Coordinator,INFORM,SCR-001-M0003,studio67.event_crisis,All mandatory operational checks have successf...
4,SCR-001-M0005,1,Operations,ALL,ACCEPT,None,studio67.event_crisis,SCR-001-O1: All mandatory operational checks h...
5,SCR-001-M0006,1,Coordinator,Finance,REQUEST,None,studio67.event_crisis,Assess SCR-001-O1
6,SCR-001-M0007,1,Finance,Coordinator,INFORM,SCR-001-M0006,studio67.event_crisis,"Committed spending 0.00 SAR + option cost 1,29..."
7,SCR-001-M0008,1,Finance,ALL,ACCEPT,None,studio67.event_crisis,SCR-001-O1: Committed spending 0.00 SAR + opti...
8,SCR-001-M0009,1,Coordinator,Safety,REQUEST,None,studio67.event_crisis,Assess SCR-001-O1
9,SCR-001-M0010,1,Safety,Coordinator,INFORM,SCR-001-M0009,studio67.event_crisis,Safety OK: True; Permit OK: True; Mitigation v...


All six ACL performatives represented by real agent/rule decisions.


## Cell 15 — ACL sequence and multi-round message rules

In [15]:
sequence=MessageBus('SCR-001')
a=sequence.send(1,'Coordinator','Operations','REQUEST','Check resources')
b=sequence.send(1,'Operations','Coordinator','INFORM','Resources assessed',in_reply_to=a.message_id)
c=sequence.send(2,'Coordinator','ALL','CFP','Request revised proposal')
assert [m.message_id for m in sequence.messages]==[
    'SCR-001-M0001','SCR-001-M0002','SCR-001-M0003']
assert b.in_reply_to==a.message_id and c.round==2
expect_error(lambda:sequence.send(1,'Coordinator','ALL','CFP','Backwards round'))
assert sequence.history(agent='Operations')==[a,b,c]
assert sequence.history(performative='INFORM')==[b]
print('ACL message ordering, IDs, broadcast, reply links, and round validation passed.')

ACL message ordering, IDs, broadcast, reply links, and round validation passed.


## Cell 16 — Integration compatibility without launching negotiation

In [16]:
from core.negotiation import NegotiationEngine
from core.runner import build_engine
assert all(hasattr(OperationsAgent,field) for field in ('name','evaluate','check_constraints'))
assert all(hasattr(MessageBus,field) for field in ('send','history','to_records'))
assert callable(NegotiationEngine) and callable(build_engine)
assert operations_agent.name=='Operations'
assert all(isinstance(e,AgentEvaluation) for e in agent_results)
assert all(isinstance(m,ACLMessage) for m in log.messages)
print('Operations V4 and ACL interfaces are compatible with the negotiation module.')

Operations V4 and ACL interfaces are compatible with the negotiation module.


## Cell 17 — Export results and evidence (no extra Gemini calls)

In [17]:
results_dir=ROOT/'results'
results_dir.mkdir(exist_ok=True)
resource_df.to_csv(results_dir/'operations_resource_gates.csv',index=False)
messages_df.to_csv(results_dir/'operations_acl_messages.csv',index=False)
pd.DataFrame([{'Test':'Feasible live',**live_approved_or_rejected.model_dump()},
              {'Test':'Mandatory failure live',**live_rejected.model_dump()}]).to_csv(
    results_dir/'operations_live_gemini_v4.csv',index=False)
pd.DataFrame([e.model_dump() for e in agent_results]).to_csv(
    results_dir/'operations_cross_agent_evaluations.csv',index=False)
print('Saved verified outputs:',results_dir)
print('No Python source files were modified.')

Saved verified outputs: /content/AgenticX/results
No Python source files were modified.
